# Hindsight Quickstart — Customer Support Example

This mirrors the structure of Hindsight's own
[quickstart notebook](https://github.com/vectorize-io/hindsight-cookbook/blob/main/notebooks/01-quickstart.ipynb),
with our own example: a customer support assistant remembering facts about a
customer, Ahmet, across separate conversations.

- **Retain**: store information in memory
- **Recall**: retrieve memories matching a query
- **Reflect**: generate an answer by reasoning over stored memories

Part 2 of this notebook goes further than the official quickstart, into how
`retain` produces the different memory types you see in the
[Admin UI](http://localhost:9999) — that part is optional, and documents a couple
of open questions we ran into.

## Prerequisites

Hindsight must already be running (see the main [README](../README.md)):

```bash
docker compose -f ../docker-compose.hindsight.yml up -d
```

## Installation

In [ ]:
%pip install --quiet hindsight-client nest_asyncio

## Connect to Hindsight

In [ ]:
# Jupyter already runs its own asyncio event loop; the Hindsight client uses
# run_until_complete() internally, and Python doesn't allow nested event loops
# by default. nest_asyncio patches this so the client works inside a notebook.
import nest_asyncio
nest_asyncio.apply()

from hindsight_client import Hindsight

HINDSIGHT_API_URL = "http://localhost:8888"
HINDSIGHT_UI_URL = "http://localhost:9999"
BANK_ID = "quickstart-demo"

client = Hindsight(base_url=HINDSIGHT_API_URL)

# Fresh start, in case this notebook has been run before.
try:
    client.delete_bank(BANK_ID)
except Exception:
    pass

## Part 1 — Quickstart

## Retain: Store Information

`retain` pushes new information into memory. Behind the scenes, an LLM extracts
structured facts, entities, and timing from the text you give it.

In [ ]:
client.retain(
    bank_id=BANK_ID,
    content="Ahmet Yilmaz, kurumsal hesabinda odeme yontemini kredi kartindan banka havalesine degistirdi.",
    context="odeme yontemi guncellemesi",
)

print(f"Dokumanlari gorebilirsin: {HINDSIGHT_UI_URL}/banks/{BANK_ID}?view=documents")

In [ ]:
# context ve timestamp ile bir kayit daha
from datetime import datetime, timezone

client.retain(
    bank_id=BANK_ID,
    content="Ahmet, aboneligini aylik plandan yillik plana yukseltti.",
    context="plan degisikligi",
    timestamp=datetime.now(timezone.utc),
)

## Recall: Retrieve Memories

`recall` retrieves memories matching a query. It searches in parallel by
meaning, keywords, entity/temporal links, and time range.

In [ ]:
results = client.recall(bank_id=BANK_ID, query="Ahmet odemeyi nasil yapiyor?")

print("Bulunanlar:")
for r in results.results:
    print(f"  - {r.text}")

In [ ]:
# Zamanla ilgili bir soru
results = client.recall(bank_id=BANK_ID, query="Bu hafta Ahmet'in hesabinda ne degisti?")

print("Bulunanlar:")
for r in results.results:
    print(f"  - {r.text}")

## Reflect: Generate an Answer

`reflect` goes further than `recall` — instead of returning raw matching facts,
it reasons over what's stored and writes an answer to your question.

In [ ]:
response = client.reflect(
    bank_id=BANK_ID,
    query="Destek ekibinin Ahmet hakkinda bilmesi gereken en onemli sey nedir?",
)
print(response.text)

## Memory Types

Hindsight organizes what it stores into a few different kinds of memory:

- **World facts** — things stated directly ("Ahmet changed his payment method")
- **Experience** — the bank's own first-person actions and interactions
- **Observations** — patterns Hindsight notices across several facts
- **Mental Models** — a standing, named question Hindsight keeps answered as new facts arrive

`retain` above created World Facts. Part 2 below digs into the other three —
skip it if you just wanted the basics.

## Cleanup

Delete the bank created during Part 1, so re-running this notebook starts clean.
(Skip this cell if you want to move on to Part 2 first — it uses the same bank.)

In [ ]:
# client.delete_bank(BANK_ID)
# print("Bank silindi.")
print("Not: Part 2'ye geciyorsan bu hucreyi calistirma, ayni bank_id kullaniliyor.")

---

## Part 2 (bonus) — what's under the hood

This part is not in the official quickstart. It looks at how `retain` above
actually produced the World Facts we saw, and tries to produce the other three
memory types. Two of the three don't fully work as you might expect — that's
shown here honestly, not glossed over.

### Observations

A single fact rarely becomes an Observation — Observations are Hindsight's own
synthesis across *multiple* facts. We add one more fact, then trigger a
consolidation pass by hand (this normally also happens automatically on a
schedule; triggering it here just makes the timing predictable for a
walkthrough). The official client doesn't expose this — it's a raw REST call.

In [ ]:
import requests
import time

client.retain(
    bank_id=BANK_ID,
    content="Ahmet, gecen ay yasadigi fatura hatasi icin destek ekibinden ozur e-postasi aldi.",
    context="fatura sikayeti",
)

requests.post(f"{HINDSIGHT_API_URL}/v1/default/banks/{BANK_ID}/consolidate").raise_for_status()

# Observation olusana kadar birkac saniye arayla kontrol et.
# Not: list_memories() sonuclari duz dict olarak donuyor (nesne degil),
# ondan "text" anahtariyla erisiyoruz.
observation = None
for _ in range(30):
    obs = client.list_memories(bank_id=BANK_ID, type="observation")
    if obs.items:
        observation = obs.items[0]
        break
    time.sleep(3)

print(observation["text"] if observation else "30 saniyede olusmadi — Admin UI'dan elle kontrol et.")

### Mental Models

A Mental Model is different: it's not produced automatically. You define it
once, as a named standing question, and Hindsight keeps an answer for it,
refreshed on demand (or automatically after consolidation).

In [ ]:
client.create_mental_model(
    bank_id=BANK_ID,
    id="ahmet-ozet",
    name="Ahmet Ozeti",
    source_query="Ahmet hakkinda bildigimiz her seyi ozetle: odeme tercihi, plan, gecmis sikayetler.",
)
client.refresh_mental_model(bank_id=BANK_ID, mental_model_id="ahmet-ozet")

content = None
for _ in range(30):
    mm = client.get_mental_model(bank_id=BANK_ID, mental_model_id="ahmet-ozet")
    if mm.content and mm.content != "Generating content...":
        content = mm.content
        break
    time.sleep(3)

print(content or "30 saniyede olusmadi.")

**Honest note:** in our testing, this sometimes comes back saying it found no
relevant information, even though the World Facts above clearly exist in the
same bank. If that happens here too, it's a real, reproducible behavior we ran
into — not a mistake in this notebook.

### Experience — an open question

The Admin UI describes Experience as *"the bank's own actions, interactions,
and first-person experiences."* We tried three ways to populate it and none
worked — all three landed as World Facts instead:

1. A `retain` call phrased as the agent's own first-person action.
2. A `retain` call shaped like a two-sided conversation transcript.
3. Uploading that same transcript through the Admin UI's **+ Add Document**.

The cell below repeats attempt 1, so you can see the (non-)result directly.

In [ ]:
client.retain(
    bank_id=BANK_ID,
    content="Bugun Ahmet ile gorustum, odeme yontemi degisikligini onayladim ve hesabini guncelledim.",
    context="temsilci notu",
)

experience = client.list_memories(bank_id=BANK_ID, type="experience")
print(f"Experience: {len(experience.items)} kayit")

world = client.list_memories(bank_id=BANK_ID, type="world")
print(f"World Facts: {len(world.items)} kayit (buraya gitti)")

If you find what actually triggers Experience, update this cell and note —
that's genuinely useful for the team.

### Real cleanup

In [ ]:
client.delete_bank(BANK_ID)
client.close()
print("Bank silindi, baglanti kapatildi.")